# Activity Notebook: Comparing Two Groups

**The business question.** A new page, a campaign or a training programme has been tried. Did it make a real difference, or are we looking at noise? And when the same people are measured twice, how do we avoid throwing away the best evidence we have?

In this notebook you reproduce, in Python, comparisons you run in JMP. JMP is the ground truth. Python has to agree with it.

> **Copilot is on.** It can write code for you in this notebook. Treat what it writes as a suggestion and check every result against the JMP output before you believe it. If the two disagree, the code is wrong until shown otherwise.

**How this notebook works**

- Parts starting **Demo:** are worked examples. Run the cells in order with **Shift+Enter** and read each result.
- Parts starting **Exercise:** are yours. You change a few lines, or write a prompt, and run the cells again.
- Every part follows the same five steps: **load and look, run, read, check against JMP, say it to a manager**.
- The data files are in `../data/`, already in your repository.

### Setup

In [ ]:
# pandas holds the tables, scipy runs the tests
import pandas as pd
from scipy import stats
from pathlib import Path

# Confirm the data folder is where this notebook expects it
print("Data files found:", sorted(p.name for p in Path("../data").iterdir()))

---
## Demo: A/B test

A web team tried a new page design (version B) against the current one (version A). 100 visitors each saw one version. The measure is minutes spent on the page. Did version B make a real difference?

### 1. Load and look

In [ ]:
# One row per visitor: which version they saw and how long they stayed
ab = pd.read_csv("../data/ab_test_website.csv")
ab.head()

In [ ]:
# How many visitors saw each version, and what are the average and spread?
ab.groupby("version")["time_on_page_min"].agg(["count", "mean", "std"]).round(3)

The two spreads (the `std` column) are close. That matters in step 2, when we choose which test to run.

### 2. Run the test

In [ ]:
# Pull out the minutes for each version. B goes first so the gap reads "B minus A".
a = ab[ab["version"] == "A"]["time_on_page_min"]
b = ab[ab["version"] == "B"]["time_on_page_min"]

# Welch's t-test: compares two averages and lets each group have its own spread
result = stats.ttest_ind(b, a, equal_var=False)

gap = b.mean() - a.mean()                 # the difference, in minutes
se = gap / result.statistic               # how much that gap would wobble from sample to sample
low, high = result.confidence_interval()  # 95% interval for the true gap

print(f"Gap (B minus A): {gap:.3f} minutes, about {gap * 60:.0f} seconds")
print(f"Standard error of the gap: {se:.3f}")
print(f"t ratio: {result.statistic:.2f}   df: {result.df:.1f}   p-value: {result.pvalue:.4f}")
print(f"95% interval for the gap: {low:.3f} to {high:.3f} minutes")

In [ ]:
# Same data with the pooled test, which assumes both groups have the same spread
pooled = stats.ttest_ind(b, a, equal_var=True)
print(f"Welch p-value: {result.pvalue:.4f}   Pooled p-value: {pooled.pvalue:.4f}")

### 3. Read it

| Output | What it says in plain English |
|---|---|
| Gap (B minus A) 0.680 | Visitors on version B stayed 0.68 minutes, about 41 seconds, longer on average. |
| Standard error 0.246 | A gap from a sample like this typically wobbles by about a quarter of a minute. |
| t ratio 2.77 | The gap is about 2.8 standard errors away from zero. |
| p-value 0.0067 | If the versions were really the same, a gap this big would turn up in about 7 tests in 1,000. |
| 95% interval 0.193 to 1.167 | The true gain is plausibly anywhere from about 12 to 70 seconds. Zero is not in the interval. |

The Welch and pooled p-values agree here because the two spreads are close. Welch is still the safe default, because it stays honest when they are not.

### 4. Check against JMP

In JMP, open `ab_test_website.csv`, compare time on page across the two versions, and run the t test that does not assume equal variances. Find the difference, its standard error, the t ratio, the degrees of freedom, Prob > |t| and the 95% limits. Then type JMP's p-value into the cell below.

Two things to watch. JMP may show the gap the other way round (A minus B), so the sign can flip. The size should not. And JMP rounds, so expect agreement to three or four decimal places.

In [ ]:
jmp_p = 0.0067   # Prob > |t| from the JMP output

print("Python p-value:", round(result.pvalue, 4))
print("Matches JMP:", abs(result.pvalue - jmp_p) < 0.0005)

### 5. Say it to a manager

> Version B kept visitors on the page about 41 seconds longer than version A, and the true gain is plausibly between 12 and 70 seconds. That is unlikely to be luck, so the gain looks real. Whether 41 seconds is worth the cost of shipping is a business call.

### Try this

1. Change `equal_var=False` to `True` in the test cell and run it again. What changes, and what does not?
2. Swap the order to `stats.ttest_ind(a, b, equal_var=False)`. What happens to the sign of the gap and the interval? What stays the same?

---
## Demo: paired readings

A clinic measured the blood pressure of 10 people in the morning and again in the evening. The same people, twice. Is there a real difference between the two times of day?

### 1. Load and look

In [ ]:
# One row per person: morning reading, evening reading and the difference
bp = pd.read_excel("../data/Blood Pressure by Time.xlsx")
bp

In [ ]:
# Does the Dif column really equal evening minus morning in every row?
bool((bp["BP PM"] - bp["BP AM"] == bp["Dif"]).all())

### 2. Run the test

A paired test works on each person's own difference. That removes the large differences between people and leaves only the change we care about.

In [ ]:
# Each person's rise from morning to evening
rise = bp["BP PM"] - bp["BP AM"]

# Paired t-test: tests whether the average rise is zero
paired = stats.ttest_rel(bp["BP PM"], bp["BP AM"])
low, high = paired.confidence_interval()   # 95% interval for the average rise

print(f"Average rise (evening minus morning): {rise.mean():.1f}")
print(f"Standard error of the rise: {rise.sem():.2f}")
print(f"t ratio: {paired.statistic:.2f}   df: {paired.df:.0f}   p-value: {paired.pvalue:.8f}")
print(f"95% interval for the rise: {low:.2f} to {high:.2f}")

In [ ]:
# The mistake to avoid: the same 20 readings treated as two unrelated groups
wrong = stats.ttest_ind(bp["BP PM"], bp["BP AM"], equal_var=False)

print(f"Paired p-value:      {paired.pvalue:.8f}")
print(f"Independent p-value: {wrong.pvalue:.3f}")

### 3. Read it

| Output | What it says in plain English |
|---|---|
| Average rise 17.6 | On average, evening readings were 17.6 points higher than morning readings. |
| Standard error 1.23 | The average rise would wobble by about 1.2 points from one group of 10 people to another. |
| t ratio 14.30, df 9 | The rise is about 14 standard errors away from zero. |
| p-value 0.00000017 | If evening and morning were really the same, a rise this large would almost never happen. |
| 95% interval 14.82 to 20.38 | The true average rise is plausibly between about 15 and 20 points. |
| Independent p-value 0.143 | Treat the same readings as two unrelated groups and the rise looks like noise. |

People differ from each other by a lot (the spread of readings is about 26). Each person's own rise varies far less (about 3.9). Pairing uses that.

### 4. Check against JMP

In JMP, use the Matched Pairs platform with the morning and evening readings. Find the mean difference, its standard error, the 95% limits, the t ratio, the degrees of freedom and Prob > |t|. Then run the same two columns as two separate groups and note that p-value as well.

JMP reports very small p-values as a bound, such as <.0001, so for the paired result you check the bound, not an exact match. As before, the sign may flip.

In [ ]:
jmp_independent_p = 0.143   # Prob > |t| when the readings are treated as two groups

print("Paired p-value is below 0.0001, as JMP reports:", paired.pvalue < 0.0001)
print("Independent p-value matches JMP:", abs(wrong.pvalue - jmp_independent_p) < 0.0005)

### 5. Say it to a manager

> Blood pressure was on average 17.6 points higher in the evening than in the morning, and the true rise is plausibly between 15 and 20 points. Every one of the 10 people rose. Testing the same people twice picks the rise up clearly, whereas treating the readings as two unrelated groups would have hidden it.

### Try this

Change `"BP PM"` and `"BP AM"` so the order is reversed in the paired test. What happens to the sign of the t ratio and the interval, and what stays the same?

---
## Exercise: email campaign

A retailer emailed a promotion to some customers (group `email`) and not to others (group `no_email`). Each customer has a satisfaction score. Did the email campaign make a real difference to satisfaction?

You do not need to write new code here. You point the code you have already seen at a new file.

### 1. Load and look

Change the four lines below so they describe the email campaign data. The file is `email_marketing_campaign.csv`. If you are not sure of the column names, load the file and look before you edit.

In [ ]:
# EXERCISE SETUP: change these four lines to point at the email campaign data
file = "../data/ab_test_website.csv"   # which file?
group_col = "version"                  # which column holds the group names?
value_col = "time_on_page_min"         # which column holds the measure?
new_group, old_group = "B", "A"        # the treated group first, the comparison group second

In [ ]:
# Look first: do the column names match what you typed above?
data = pd.read_csv(file)
data.head()

In [ ]:
# Group sizes, averages and spreads
data.groupby(group_col)[value_col].agg(["count", "mean", "std"]).round(3)

### 2. Run the test

In [ ]:
# Split the measure into the two groups
new = data[data[group_col] == new_group][value_col]
old = data[data[group_col] == old_group][value_col]

# Welch's t-test, then the gap, its standard error and the 95% interval
result = stats.ttest_ind(new, old, equal_var=False)
gap = new.mean() - old.mean()
se = gap / result.statistic
low, high = result.confidence_interval()

print(f"Gap ({new_group} minus {old_group}): {gap:.3f}")
print(f"Standard error of the gap: {se:.3f}")
print(f"t ratio: {result.statistic:.2f}   df: {result.df:.1f}   p-value: {result.pvalue:.4f}")
print(f"95% interval for the gap: {low:.3f} to {high:.3f}")

### 3. Read it

Write a sentence for each line of the output in the commentary cell at the end of this part.

### 4. Check against JMP

You ran this comparison in JMP in the Activity Workbook. Type the Prob > |t| from that output into the cell below.

In [ ]:
jmp_p = None # type the Prob > |t| from your JMP output here
if jmp_p is None:
    print("Type the JMP p-value into jmp_p, then run this cell again.")
else:
    print("Python p-value:", round(result.pvalue, 4))
    print("Matches JMP:", abs(result.pvalue - jmp_p) < 0.0005)

### 5. Say it to a manager

Write two sentences in the cell below. Give the gap in plain units with its interval, then say what the result does and does not let you conclude.

*Your two sentences here*

### Commentary

1. What is the gap, and what is the 95% interval for it? Does the interval include zero?
2. Which of the four setup lines did you change? Did you meet an error, and how did you fix it?

*Your commentary here*

---
## Exercise: training programme

A company ran a training programme. Some employees took it (group `yes`) and some did not (group `no`). Each has a performance score. Did the programme make a real difference?

This time you ask Copilot to write the code. Your job is to give it a clear prompt and then to check what it produces against JMP.

### 1. Load and look

The file is `training_programme.csv`. Load it and look at it first, so your prompt names the real columns.

In [ ]:
# Load the file and look at the first rows, so you know the real column names

### 2. Run the test

Write a prompt to Copilot as a comment in the cell below, then let it suggest the code. A good prompt says:

- which file to load, and which columns hold the group and the measure
- which group is the treated group, so the gap reads "treated minus comparison"
- that you want Welch's t-test for two independent groups
- what to print: the gap, the t ratio, the degrees of freedom, the p-value and the 95% interval
- that each line should have a short comment

If Copilot is not available to you, copy the cells from the email campaign exercise and change the four setup lines instead.

In [ ]:
# Prompt to Copilot:

### 3. Read it

What does each printed number say? Write your answer in the commentary cell at the end of this part. Pay attention to what the interval does and does not include.

### 4. Check against JMP

Type the Prob > |t| from your JMP output into the cell below.

In [ ]:
jmp_p = None   # type the Prob > |t| from your JMP output here

if jmp_p is None:
    print("Type the JMP p-value into jmp_p, then run this cell again.")
else:
    print("Python p-value:", round(result.pvalue, 4))
    print("Matches JMP:", abs(result.pvalue - jmp_p) < 0.0005)

### Find the bug

Here is code an assistant might write for the same question. It runs without an error. Run it, then compare what it prints with your JMP output.

In [ ]:
# Code an assistant might write for the training question
training = pd.read_csv("../data/training_programme.csv")
trained = training[training["training"] == "Yes"]["performance_score"]
untrained = training[training["training"] == "No"]["performance_score"]

buggy = stats.ttest_ind(trained, untrained, equal_var=False)
print(f"t ratio: {buggy.statistic:.2f}   p-value: {buggy.pvalue:.4f}")

### 5. Say it to a manager

Write two sentences in the cell below. Give the gap in plain units with its interval, then say what the result does and does not let you conclude.

*Your two sentences here*

### Commentary

1. Why does the interval include zero, and what does that not tell you?
2. What did the buggy cell print, how did the JMP result show you it was wrong, and what was the fix?

*Your commentary here*

---
## Say it to a manager

Choose the last result you produced. In the cell below, write two sentences a manager could act on:

1. The gap in plain units, with the interval.
2. What the result does and does not let you conclude.

No jargon. If a sentence needs the words "t ratio", rewrite it.

*Your two sentences here*

---
## Stretch

Finished early? Use the blank space below. Pick one of these and vibe-code it with Copilot, checking each answer against what you already know from JMP.

- Test whether the values in each group look roughly bell-shaped (try the Shapiro-Wilk test in scipy), and say whether it matters for the result.
- Write a short function that takes a file name and two column names, and prints the whole comparison in one go.
- Build a bootstrap interval for the gap (resample each group many times) and compare it with the interval from the test.

In [ ]:
# Stretch